In [ ]:
from google.colab import files
files.upload()
!mkdir -p ~/.kaggle
!cp kaggle.json ~/.kaggle/
!chmod 600 ~/.kaggle/kaggle.json

Saving kaggle.json to kaggle.json


In [ ]:
import kagglehub
import shutil

# Download latest version
path = kagglehub.dataset_download("patelris/crop-yield-prediction-dataset")
destination = "/content/crop-yield-prediction-dataset"
shutil.copytree(path, destination, dirs_exist_ok=True)

print("Path to dataset files:", path)

Using Colab cache for faster access to the 'crop-yield-prediction-dataset' dataset.
Path to dataset files: /kaggle/input/crop-yield-prediction-dataset


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

data = pd.read_csv("/content/crop-yield-prediction-dataset/yield_df.csv")
df = pd.DataFrame(data)
df.head()

,Unnamed: 0,Area,Item,Year,hg/ha_yield,average_rain_fall_mm_per_year,pesticides_tonnes,avg_temp
0,0,Albania,Maize,1990,36613,1485.0,121.0,16.37
1,1,Albania,Potatoes,1990,66667,1485.0,121.0,16.37
2,2,Albania,"Rice, paddy",1990,23333,1485.0,121.0,16.37
3,3,Albania,Sorghum,1990,12500,1485.0,121.0,16.37
4,4,Albania,Soybeans,1990,7000,1485.0,121.0,16.37


In [ ]:
#demo cell
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
gs=GridSearchCV(Ridge(),{"alpha":[0.01,0.1,1,10,100]},cv=5).fit(Xtr,ytr)
print(gs.best_params_)

In [ ]:
yield_crops = df['hg/ha_yield'].to_numpy()
rainfall = df['average_rain_fall_mm_per_year'].to_numpy()

idx=np.random.default_rng(0).permutation(len(yield_crops))
cut=int(0.8*len(yield_crops)); tr,te=idx[:cut],idx[cut:]



In [ ]:
#select training data
x = rainfall[tr]
y = yield_crops[tr]

#standardise training data
x_mean, x_std = x.mean(), x.std()
y_mean, y_std = y.mean(), y.std()

Xtr = ((x - x_mean) / x_std).reshape(-1,1)
ytr = (y - y_mean) / y_std


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV
gs=GridSearchCV(Ridge(),{"alpha":[0.01,0.1,1,10,100]},cv=5).fit(Xtr,ytr)
print(gs.best_params_)

{'alpha': 100}


In [ ]:
Xte = ((rainfall[te] - x_mean) / x_std).reshape(-1, 1)
yte = (yield_crops[te] - y_mean) / y_std

#prediction
y_pred = gs.best_estimator_.predict(Xte)

# calculate test mse
test_mse = ((yte - y_pred)**2).mean()

print("Test MSE:", test_mse)
print("Test RMSE:", np.sqrt(test_mse))

Test MSE: 1.0328152939691735
Test RMSE: 1.0162752058223075


In [ ]:
feature = df[['average_rain_fall_mm_per_year', 'avg_temp', 'pesticides_tonnes']].to_numpy()

# select training data
x = feature[tr]
y = yield_crops[tr]

# standardise training data
x_mean = x.mean(axis=0)
x_std = x.std(axis=0)

y_mean, y_std = y.mean(), y.std()

Xtr = (x - x_mean) / x_std
ytr = (y - y_mean) / y_std

# prepare test data
Xte = (feature[te] - x_mean) / x_std
yte = (yield_crops[te] - y_mean) / y_std

In [ ]:
lr = 0.0001
alphas = [0, 0.01, 0.1, 1, 10, 100]

for alpha in alphas:
    w = np.zeros(Xtr.shape[1])
    b = 0.0

    for e in range(2000):
        err = Xtr @ w + b - ytr
        w -= lr * (2 * (Xtr.T @ err) / len(ytr) + 2 * alpha * w)
        b -= lr * 2 * err.mean()

    #test pred
    y_pred = Xte @ w + b

    # Test MSE
    mse = ((yte - y_pred)**2).mean()

    print(f"Alpha: {alpha:6} | Test MSE: {mse:.6f}")

Alpha:      0 | Test MSE: 1.021812
Alpha:   0.01 | Test MSE: 1.021829
Alpha:    0.1 | Test MSE: 1.021982
Alpha:      1 | Test MSE: 1.023373
Alpha:     10 | Test MSE: 1.029416
Alpha:    100 | Test MSE: 1.032345


In [ ]:
from sklearn.linear_model import Ridge
from sklearn.model_selection import GridSearchCV

gs = GridSearchCV(
    Ridge(),
    {"alpha": [0, 0.001, 0.01, 0.1, 1, 10, 100]},
    cv=5,
    scoring='neg_mean_squared_error'
)

gs.fit(Xtr, ytr)

print("Best Alpha:", gs.best_params_)
print("Best CV MSE:", -gs.best_score_)

y_pred = gs.best_estimator_.predict(Xte)
print("Test MSE:", ((yte - y_pred)**2).mean())

Best Alpha: {'alpha': 100}
Best CV MSE: 0.9833045749033577
Test MSE: 1.0087259469331855
